## MCORR AND CNMF
**Analysis of riboL1-GCaMP8m responses imaged with 16x objective at zoom 1x over 765x765 pixels**

#### Define parameters


In [4]:
# Paths

from pathlib import Path
data_path = [Path('D:/Data/2P/C57_O1M2/10272023/TSeries-10272023-1335-009')]
export_path = Path('H:/Analysis/2P/C57_O1M2/10272023/run9/mesmerize/')

# Motion correction parameters

params_mcorr = \
{
  'main':
    {
        'strides': [36, 36],
        'overlaps': [24, 24],
        'max_shifts': [12, 12],
        'max_deviation_rigid': 6,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# CNMF parameters
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 20, # framerate, very important!
            'p': 0,
            'nb': 1,
            'merge_thr': 0.8,
            'rf': 20,
            'stride': 10, # "stride" for cnmf, "strides" for mcorr
            'K': 12,
            'gSig': [4, 4],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.5,
            'SNR_lowest':  1.5,
            'rval_thr': 0.85,
            'rval_lowest': 0.2,
            'use_cnn': True,
            'min_cnn_thr': 0.95,
            'cnn_lowest': 0.2,
            'decay_time': 0.10,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

# Extra parameters
params_extra = \
    {
        'cleanup': False # If `True`, run cleanup after CNMF, i.e., delete all df data
    }

# Concatenate the two dictionaries
params = {'params_mcorr': params_mcorr, 'params_cnmf': params_cnmf, 'params_extra': params_extra}
print(params)

{'params_mcorr': {'main': {'strides': [36, 36], 'overlaps': [24, 24], 'max_shifts': [12, 12], 'max_deviation_rigid': 6, 'border_nan': 'copy', 'pw_rigid': True, 'gSig_filt': None}}, 'params_cnmf': {'main': {'fr': 20, 'p': 0, 'nb': 1, 'merge_thr': 0.8, 'rf': 20, 'stride': 10, 'K': 12, 'gSig': [4, 4], 'ssub': 1, 'tsub': 1, 'method_init': 'greedy_roi', 'min_SNR': 2.5, 'SNR_lowest': 1.5, 'rval_thr': 0.85, 'rval_lowest': 0.2, 'use_cnn': True, 'min_cnn_thr': 0.95, 'cnn_lowest': 0.2, 'decay_time': 0.1}, 'refit': True}, 'params_extra': {'cleanup': False}}


#### Run MCORR and CNMF

In [5]:
import pipeline_mcorr_cnmf as preproc
_ , batch_path = preproc.run_mcorr_cnmf(data_path, params, export_path)

Starting run_mcorr_cnmf pipeline.
Loading and concatenating data from [WindowsPath('D:/Data/2P/C57_O1M2/10272023/TSeries-10272023-1335-009')].
Found 1600 files to concatenate.
Saving concatenated file to directory: H:\Analysis\2P\C57_O1M2\10272023\run9\mesmerize
Images are uint16 with range (0, 4095).
Concatenated 1600 files to multi-page TIFF H:\Analysis\2P\C57_O1M2\10272023\run9\mesmerize\cat_tiff_mpt.tiff.
Images are uint16 with range 0 - 4095.
Set -r flag to False to disable conversion from uint12 to uint16.
Converted data range: 0 - 65535
Removed H:\Analysis\2P\C57_O1M2\10272023\run9\mesmerize\cat_tiff_mpt.tiff
Concatenated 1600 files to BigTIFF H:\Analysis\2P\C57_O1M2\10272023\run9\mesmerize\cat_tiff_bt.tiff.
Concatenation completed in 10.89 seconds.
Concatenated movie path: H:\Analysis\2P\C57_O1M2\10272023\run9\mesmerize\cat_tiff_bt.tiff.
Creating batch H:\Analysis\2P\C57_O1M2\10272023\run9\mesmerize\batch_20240118-191926.pickle.
Running batch item 0, id c41eca60-2ac2-4df6-8703-

### Load outputs

In [6]:
# If cleanup was set to false in the params, you can load the batch file into Mesmerize:
# batch_path=Path('H:/Analysis/2P/C57_O1M2/10022023/run7/mesmerize/batch_20231230-135733.pickle')
if params['params_extra']['cleanup'] == False:
    import mesmerize_core as mc
    df = mc.load_batch(batch_path)

### Visualize with fastplotlib


In [7]:
import fastplotlib as fpl
cnmf_index = 1
rcm = df.iloc[cnmf_index].cnmf.get_rcm()
rcb = df.iloc[cnmf_index].cnmf.get_rcb()
residuals = df.iloc[cnmf_index].cnmf.get_residuals()
input_movie = df.iloc[cnmf_index].caiman.get_input_movie()

iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

RFBOutputContext()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\graphics\_features\_base.py:34: UserWarning: converting float64 array to float32
  warn(f"converting {array.dtype} array to float32")


JupyterOutputContext(children=(JupyterWgpuCanvas(css_height='600px', css_width='800px'), IpywidgetToolBar(chil…

In [9]:
iw_rcm.close()

### Visualize with `mesmerize-viz`

In [8]:
print(f"Num accepted/rejected: {len(df.iloc[-1].cnmf.get_good_components())}, {len(df.iloc[-1].cnmf.get_bad_components())}")

Num accepted/rejected: 1437, 4560


In [10]:
import mesmerize_viz
viz_simple = df.cnmf.viz(
    image_data_options=["max"],
)
viz_simple.show(sidecar=True)
# viz_simple.show()
viz_simple.image_widget.cmap = "gray"

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\ipydatagrid\datagrid.py:460: UserWarning: Index name of 'index' is not round-trippable.
  schema = pd.io.json.build_table_schema(dataframe)


RFBOutputContext()

RFBOutputContext()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\graphics\_features\_base.py:34: UserWarning: converting float64 array to float32
  warn(f"converting {array.dtype} array to float32")


RFBOutputContext()

In [12]:
viz_simple.close()

#### Clean up export folder: stop/restart notebook, then run first and last cells

In [2]:
import pipeline_mcorr_cnmf as preproc
batch_path=Path('H:/Analysis/2P/C57_O1M2/10272023/run9/mesmerize/batch_20240118-183236.pickle')
preproc.cleanup_files(batch_path, export_path)

Batch files deleted.
